# Session 8: pandas

Nothing in this session is a new idea. Every question here is one you have
already answered, with a loop in session 4 and with SQL in sessions 6 and 7.
What is new is the notation. Watch for the moments where it maps across
one to one, and for the two places where it does not.

In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

# Walk up to the repository root, whatever folder the notebook was opened from.
here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

print("pandas", pd.__version__)

`import pandas as pd` is how everybody writes it. The `pd` is only a
nickname, so `pd.read_csv` means "the `read_csv` from pandas". You will see
`pd.` in every pandas example on the internet.

## A DataFrame is the shape you already know

In session 4, a list of dictionaries was a table: one dictionary per row,
one key per column. Hand that exact shape to pandas.

In [ ]:
rows = [
    {"genre": "Folk", "minutes": 4.65},
    {"genre": "Jazz", "minutes": 7.10},
    {"genre": "Folk", "minutes": 2.42},
]

small = pd.DataFrame(rows)
small

The same three rows, now drawn as a table. The numbers down the left, 0 to
2, are the **index**: pandas' own row labels. Ignore them for now.

A **DataFrame** is a list of dicts with a much faster engine underneath, and
a few hundred methods for the things you kept writing loops for.

## Loading the listening log

Compare this with session 4: no `open()`, no `encoding`, no `list()` around a
reader.

In [ ]:
plays = pd.read_csv("data/clean/plays.csv")
plays.head()

`head()` shows the first five rows. `head(10)` shows ten. It is the
`rows[:3]` peek from session 4, and you will type it constantly.

## The first four things to look at

On any new table, before asking anything clever:

In [ ]:
plays.shape

`(2183, 9)`: rows, then columns. No brackets after `shape`, because it is a
fact about the table rather than something it does.

In [ ]:
plays.info()

`info()` is the most useful single look. Every column, how many values are
filled in ("non-null"), and the **dtype**: what kind of value the column
holds.

In [ ]:
plays.dtypes

Look at `minutes_played`: **`float64`**. In session 4 every value from the
CSV arrived as text, and `total += row["minutes_played"]` was a `TypeError`.
`read_csv` looked at the column, saw numbers, and converted it for you.
`skipped` is `int64`, a whole number, so you no longer compare it with the
text `"1"`.

`object` means text, for our purposes. Notice that `played_at` is text too:
pandas does not guess dates unless you ask. That is fine for today, because
`"2025-01-01"` style dates compare and sort correctly as text, exactly as
they did in SQL.

## Columns

One column, by name in square brackets, the way you read a dictionary key:

In [ ]:
plays["genre"].head()

One column on its own is a **Series**. A DataFrame is a set of Series that
share an index.

Several columns: give it a **list** of names. That is why there are two
pairs of brackets: the outer ones say "select", the inner ones make a list.

In [ ]:
plays[["track_name", "minutes_played"]].head(3)

In [ ]:
print(type(plays["genre"]).__name__)
print(type(plays[["genre"]]).__name__)

A Series can do arithmetic on the whole column at once. This is the
accumulate loop from session 2 and the `float()` loop from session 4, in one
word:

In [ ]:
print(plays["minutes_played"].sum())
print(round(plays["minutes_played"].sum(), 1))
print(round(plays["minutes_played"].mean(), 2))
print(plays["minutes_played"].max())

8,980.4 minutes: the same total as session 4 and as `SUM(minutes_played)` in
session 6. Getting the number you already know is the first check that you
have loaded the data properly.

## Filtering rows

Comparing a column with a value does not give one `True` or `False`. It
gives one **per row**:

In [ ]:
long_ones = plays["minutes_played"] > 9
long_ones.head()

In [ ]:
long_ones.sum()

A column of `True` and `False` is called a **mask**. Adding it up counts the
`True`s, because `True` counts as 1: 32 plays longer than nine minutes, the
same 32 as rung 3 of the session 6 ladder.

Put the mask inside the square brackets and you keep only the `True` rows.
This is `WHERE`.

In [ ]:
plays[plays["minutes_played"] > 9].head()

## Two conditions: the trap

In plain Python you would write `and`. In pandas that fails:

In [ ]:
try:
    plays[(plays["device"] == "speaker") and (plays["played_at"] >= "2025-01-01")]
except ValueError as error:
    print("ValueError:", error)

"The truth value of a Series is ambiguous." `and` wants one `True` or
`False`, and it has been given 2,183 of them. Which one did you mean?

In pandas, `and` becomes **`&`**, `or` becomes **`|`**, and `not` becomes
**`~`**. Each condition goes in its own round brackets.

In [ ]:
speaker_2025 = (plays["device"] == "speaker") & (plays["played_at"] >= "2025-01-01")
speaker_2025.sum()

174, the same as session 6 rung 4.

Why the brackets? `&` is evaluated **before** `==` and `>=`. Leave the
brackets out and Python tries `"speaker" & plays["played_at"]` first, which
makes no sense:

In [ ]:
try:
    plays[plays["device"] == "speaker" & plays["played_at"] >= "2025-01-01"]
except TypeError as error:
    print("TypeError:", error)

That one at least crashes. The nastier version is mixing `&` and `|`,
because it runs. "Long plays (over 5 minutes) on the phone or the laptop":

In [ ]:
right = (plays["minutes_played"] > 5) & ((plays["device"] == "phone") | (plays["device"] == "laptop"))
wrong = (plays["minutes_played"] > 5) & (plays["device"] == "phone") | (plays["device"] == "laptop")

print("with the extra brackets:   ", right.sum())
print("without the extra brackets:", wrong.sum())

523 against 871. `&` binds tighter than `|`, so the second line means "long
plays on the phone, **or any play at all on the laptop**". No error, no
warning, a plausible number. SQL has the same rule about `AND` and `OR`; it
is the same trap in a new notation.

When a condition is "this column is one of these values", `isin` avoids the
`|` entirely:

In [ ]:
print(((plays["device"] == "car") | (plays["device"] == "tablet")).sum())
print(plays["device"].isin(["car", "tablet"]).sum())

That is `IN ('car', 'tablet')` from session 6.

## Sorting

`sort_values` is `ORDER BY`. `ascending=False` is `DESC`.

In [ ]:
plays.sort_values("minutes_played", ascending=False).head(3)

`sort_values` **returns a new DataFrame** and leaves `plays` alone. That is
the opposite of `list.sort()` in session 4, which changed the list and
returned `None`. So here, a line that sorts and does not assign the result
does nothing you can see later:

In [ ]:
plays.sort_values("minutes_played", ascending=False)   # sorted, then thrown away
plays.head(2)                                           # still in the original order

Almost every pandas method works this way: it hands you a new table. If you
want to keep the result, give it a name.

## Counting: `value_counts`

The question from session 4, slide 19, and session 6:

In [ ]:
plays["genre"].value_counts()

Same idea, three notations:

| Session 4, a loop | Session 6, SQL | Session 8, pandas |
|---|---|---|
| `counts[g] = counts.get(g, 0) + 1` | `SELECT genre, COUNT(*) ... GROUP BY genre` | `plays["genre"].value_counts()` |

`value_counts` sorts biggest first on its own. Electronic 547, Folk 438: the
same numbers all three times.

## Grouping: `groupby`

`value_counts` only counts. For anything else per group, there is
`groupby`: split the rows by genre, then do something to each group.

In [ ]:
plays.groupby("genre")["minutes_played"].sum()

Read it left to right: take `plays`, group by `genre`, take the
`minutes_played` column, add it up. Notice the genres come out in
alphabetical order, not by size: `groupby` sorts by the group, and you sort
by the value yourself.

Several numbers per group at once is `agg`, with one line per new column:
`name=(column, what_to_do)`.

In [ ]:
per_genre = plays.groupby("genre").agg(
    plays=("play_id", "count"),
    minutes=("minutes_played", "sum"),
    avg_min=("minutes_played", "mean"),
)
per_genre.sort_values("avg_min", ascending=False).round(2)

Jazz has the longest average play at 6.57 minutes, despite having the
second fewest plays: your session 4 homework, with its two dictionaries,
in one statement.

The genre has become the **index**, the row labels on the left. Most of the
time that is what you want. If you ever need it back as an ordinary column,
`.reset_index()` does that.

Filtering the groups afterwards is just filtering again. This is `HAVING`:

In [ ]:
per_genre[per_genre["plays"] > 300]

In SQL, `WHERE` and `HAVING` are two keywords because the database needs to
know whether to filter before or after grouping. In pandas you simply write
the filter on the line before the `groupby`, or on the line after it. The
order of your lines **is** the order things happen.

## Reading from the database

Everything so far came from the CSV. `pd.read_sql` runs a query and hands
back the answer as a DataFrame.

In [ ]:
con = sqlite3.connect("data/music.db")

db_plays = pd.read_sql("SELECT * FROM plays", con)
artists = pd.read_sql("SELECT * FROM artists", con)

print(db_plays.shape, artists.shape)
db_plays.head(3)

Same 2,183 plays, but this is the database version: `artist_id` instead of
the artist's name, genre and country, because those live in `artists`.

The query can be any SQL you like, so you can let the database do the
grouping and only bring back the answer:

In [ ]:
pd.read_sql("SELECT device, COUNT(*) AS plays FROM plays GROUP BY device ORDER BY plays DESC", con)

`read_sql` is the bridge between the two halves of this course. SQL to fetch,
pandas to work with what came back.

## Joining: `merge`

To get names onto the database plays, join the two tables. In pandas a join
is called a **merge**.

In [ ]:
named = db_plays.merge(artists, on="artist_id")

print(len(db_plays), "rows before,", len(named), "rows after")
named[["played_at", "artist_name", "track_name", "country"]].head()

`on="artist_id"` is `USING (artist_id)`. By default `merge` is an **inner**
join: only rows that match on both sides survive.

And the habit from session 7, first thing after any join: **compare the
row count**. 2,183 before, 2,183 after. Every play found exactly one artist,
which is what should happen when you join many plays to one artist each.

`how="left"` is `LEFT JOIN`. Put `artists` on the left and the two artists
you have never played come along too:

In [ ]:
everyone = artists.merge(db_plays, on="artist_id", how="left")
print(len(everyone), "rows")

everyone[everyone["play_id"].isna()][["artist_name", "genre", "play_id"]]

2,185 rows: the 2,183 plays, plus one row each for Iron Fernway and Gravel
Choir, with nothing in the play columns. pandas writes "nothing" as
**`NaN`**, "not a number", which is its version of SQL's `NULL`. And as with
`NULL`, you ask for it with a method, `isna()`, rather than with `==`.

Those two extra rows are exactly where session 7's `COUNT(*)` bug lived.
The next notebook shows that pandas has the same bug, under a different
name.

In [ ]:
con.close()

## What you have seen

| Idea | SQL | pandas |
|---|---|---|
| load | `SELECT * FROM plays` | `pd.read_csv(...)`, `pd.read_sql(...)` |
| some columns | `SELECT track_name, minutes_played` | `plays[["track_name", "minutes_played"]]` |
| filter rows | `WHERE a AND b` | `plays[(a) & (b)]` |
| one of several | `IN ('car', 'tablet')` | `.isin(["car", "tablet"])` |
| sort | `ORDER BY minutes_played DESC` | `.sort_values("minutes_played", ascending=False)` |
| first few | `LIMIT 5` | `.head(5)` |
| count per group | `GROUP BY genre` with `COUNT(*)` | `["genre"].value_counts()` |
| several numbers per group | `GROUP BY` with `SUM`, `AVG` | `.groupby("genre").agg(...)` |
| filter groups | `HAVING` | a filter after the `groupby` |
| join | `JOIN ... USING (artist_id)` | `.merge(..., on="artist_id")` |
| missing | `IS NULL` | `.isna()` |

## Next

`02-sql-and-pandas.ipynb`: four queries from sessions 6 and 7, each written
both ways, with a check that both give the same answer.